# Classifier 1

In [1]:
from pathlib import Path
import warnings
from functools import reduce
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import f_oneway, kruskal, ttest_ind
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

INDEX_PATH = Path("index_data.csv")
RETURN_PATH = Path("return.csv")
OUTPUT_PATH = Path("classifier1_full_sample_predictions.csv")

MAIN_CODES = ["000001", "399001", "000020", "000905", "399005", "000852"]
GROWTH_CODES = ["000688", "399006", "399673"]
RISK_CODES = ["399006", "000688", "000852"]
TRAIN_END = 20241231
HOLDOUT_START = 20250101
HOLDOUT_END = 20251231
ANN_FACTOR = 242
EPS = 1e-12
OOF_FOLDS = 4
FINAL_CONFIG = {
    "label_method": "ternary_threshold",
    "upper_thresh": 0.009,
    "lower_thresh": -0.005,
    "clf_type": "lgbm",
}


In [2]:
def rolling_zscore(series: pd.Series, window: int = 252, min_periods: int = 60) -> pd.Series:
    mu = series.rolling(window, min_periods=min_periods).mean()
    sd = series.rolling(window, min_periods=min_periods).std()
    return (series - mu) / sd.replace(0, np.nan)


def calc_max_drawdown(ret_series: pd.Series) -> float:
    if len(ret_series) == 0:
        return np.nan
    nav = (1 + ret_series.fillna(0)).cumprod()
    peak = nav.cummax()
    return float((nav / peak - 1).min())


def calc_tstat(x: pd.Series) -> float:
    x = pd.Series(x).dropna()
    if len(x) < 2:
        return np.nan
    s = x.std()
    if pd.isna(s) or s < EPS:
        return np.nan
    return float(x.mean() / (s / np.sqrt(len(x))))


def summarize_strategy_by_state(df: pd.DataFrame, state_col: str = "state", ret_col: str = "ret") -> pd.DataFrame:
    rows = []
    total = len(df[ret_col].dropna()) if ret_col in df.columns else 0
    for state, g in df.groupby(state_col):
        r = g[ret_col].dropna()
        if len(r) == 0:
            continue
        mean_r = float(r.mean())
        std_r = float(r.std())
        rows.append({
            "state": int(state),
            "count": int(len(r)),
            "count_pct": len(r) / total if total else np.nan,
            "mean_ret": mean_r,
            "std_ret": std_r,
            "tstat": calc_tstat(r),
            "sharpe_ann": mean_r / std_r * np.sqrt(ANN_FACTOR) if std_r > EPS else np.nan,
            "win_rate": float((r > 0).mean()),
            "max_drawdown": calc_max_drawdown(r),
        })
    return pd.DataFrame(rows).sort_values("state").reset_index(drop=True)


def state_return_hypothesis_tests(df: pd.DataFrame, state_col: str = "state", ret_col: str = "ret") -> Dict[str, Any]:
    work = df[[state_col, ret_col]].dropna().copy()
    states = sorted(work[state_col].unique().tolist())
    groups = [work.loc[work[state_col] == s, ret_col].dropna() for s in states]
    tests = {
        "anova_p": np.nan,
        "kruskal_p": np.nan,
        "pairwise": pd.DataFrame(columns=["state_i", "state_j", "mean_i", "mean_j", "mean_diff", "t_stat", "p_value"]),
    }
    if len(groups) >= 2 and all(len(g) > 1 for g in groups):
        tests["anova_p"] = float(f_oneway(*groups).pvalue)
        tests["kruskal_p"] = float(kruskal(*groups).pvalue)
        pairwise = []
        for i, s_i in enumerate(states):
            for j, s_j in enumerate(states[i + 1 :], start=i + 1):
                g_i, g_j = groups[i], groups[j]
                tt = ttest_ind(g_i, g_j, equal_var=False, nan_policy="omit")
                pairwise.append({
                    "state_i": int(s_i),
                    "state_j": int(s_j),
                    "mean_i": float(g_i.mean()),
                    "mean_j": float(g_j.mean()),
                    "mean_diff": float(g_i.mean() - g_j.mean()),
                    "t_stat": float(tt.statistic),
                    "p_value": float(tt.pvalue),
                })
        tests["pairwise"] = pd.DataFrame(pairwise)
    return tests


In [3]:
def build_single_index_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values("trade_date").reset_index(drop=True)
    df["r1"] = df["close"] / df["preclose"].replace(0, np.nan) - 1
    for w in [3, 5, 10, 20, 60]:
        df[f"r{w}"] = df["close"] / df["close"].shift(w).replace(0, np.nan) - 1
    df["amp"] = df["high"] / df["low"].replace(0, np.nan) - 1
    df["intraday_ret"] = df["close"] / df["open"].replace(0, np.nan) - 1
    df["gap"] = df["open"] / df["preclose"].replace(0, np.nan) - 1
    for w in [5, 20, 60]:
        df[f"vol_{w}"] = df["r1"].rolling(w, min_periods=max(3, w // 3)).std()
    for w in [20, 60]:
        df[f"ma{w}"] = df["close"].rolling(w, min_periods=max(5, w // 3)).mean()
        df[f"ma{w}_bias"] = df["close"] / df[f"ma{w}"].replace(0, np.nan) - 1
    df["ma20_over_ma60"] = df["ma20"] / df["ma60"].replace(0, np.nan) - 1
    for w in [20, 60]:
        df[f"liq_z_{w}"] = rolling_zscore(df["volume"], w, max(10, w // 3))
    df["up_frac_10"] = df["r1"].gt(0).rolling(10, min_periods=5).mean()
    df["up_frac_20"] = df["r1"].gt(0).rolling(20, min_periods=10).mean()
    df["dist_from_high_20"] = df["close"] / df["high"].rolling(20, min_periods=10).max().replace(0, np.nan) - 1
    df["vol_shock"] = df["vol_5"] / df["vol_20"].replace(0, np.nan) - 1
    return df


def build_market_feature_table(df_raw: pd.DataFrame) -> pd.DataFrame:
    pieces = []
    for code, g in df_raw.groupby("idx"):
        x = build_single_index_features(g)
        x["idx"] = str(code % 1000000).zfill(6)
        pieces.append(x)
    feat_long = pd.concat(pieces, axis=0, ignore_index=True)
    keep_cols = [
        "trade_date", "idx", "r1", "r5", "r20", "r60", "vol_5", "vol_20", "vol_60",
        "ma20_bias", "ma60_bias", "ma20_over_ma60", "liq_z_20", "up_frac_10", "up_frac_20",
        "dist_from_high_20", "vol_shock",
    ]
    feat_long = feat_long[keep_cols].copy()
    wide = []
    for idx_name, g in feat_long.groupby("idx"):
        gg = g.drop(columns=["idx"]).copy()
        wide.append(gg.rename(columns={c: f"{idx_name}__{c}" for c in gg.columns if c != "trade_date"}))
    feat = reduce(lambda l, r: pd.merge(l, r, on="trade_date", how="outer"), wide)
    feat = feat.sort_values("trade_date").reset_index(drop=True)

    def existing_cols(codes: List[str], suffix: str) -> List[str]:
        return [f"{code}__{suffix}" for code in codes if f"{code}__{suffix}" in feat.columns]

    for suffix in ["r1", "r5", "r20", "r60", "vol_5", "vol_20", "vol_60", "ma20_bias", "ma60_bias", "up_frac_10", "up_frac_20"]:
        main_cols = existing_cols(MAIN_CODES, suffix)
        growth_cols = existing_cols(GROWTH_CODES, suffix)
        feat[f"main_{suffix}"] = feat[main_cols].mean(axis=1) if main_cols else np.nan
        feat[f"growth_{suffix}"] = feat[growth_cols].mean(axis=1) if growth_cols else np.nan
        feat[f"style_spread_{suffix}"] = feat[f"main_{suffix}"] - feat[f"growth_{suffix}"]

    for suffix in ["r1", "r5", "r20", "r60"]:
        risk_cols = existing_cols(RISK_CODES, suffix)
        base_col = f"000001__{suffix}" if f"000001__{suffix}" in feat.columns else None
        feat[f"risk_on_{suffix}"] = feat[risk_cols].mean(axis=1) - feat[base_col] if base_col and risk_cols else np.nan

    all_r1_cols = [c for c in feat.columns if c.endswith("__r1")]
    all_r5_cols = [c for c in feat.columns if c.endswith("__r5")]
    all_ma20_bias_cols = [c for c in feat.columns if c.endswith("__ma20_bias")]
    feat["breadth_up_ratio_1"] = feat[all_r1_cols].gt(0).mean(axis=1)
    feat["breadth_up_ratio_5"] = feat[all_r5_cols].gt(0).mean(axis=1)
    feat["breadth_trend_ratio"] = feat[all_ma20_bias_cols].gt(0).mean(axis=1)
    feat["cross_dispersion_r1"] = feat[all_r1_cols].std(axis=1)
    feat["cross_dispersion_r5"] = feat[all_r5_cols].std(axis=1)
    feat["main_vol_shock"] = feat["main_vol_5"] / feat["main_vol_20"].replace(0, np.nan) - 1
    feat["vol_spread_vol_5"] = feat["main_vol_5"] - feat["growth_vol_5"]
    feat["vol_spread_vol_20"] = feat["main_vol_20"] - feat["growth_vol_20"]
    feat["vol_spread_vol_60"] = feat["main_vol_60"] - feat["growth_vol_60"]

    for code in ["399006", "000688", "000905", "000852", "399001"]:
        for suffix in ["r1", "r5", "r20"]:
            left, right = f"{code}__{suffix}", f"000001__{suffix}"
            if left in feat.columns and right in feat.columns:
                feat[f"{code}_minus_000001_{suffix}"] = feat[left] - feat[right]

    feat["growth_lead_r5"] = feat["growth_r5"] - feat["main_r5"]
    feat["growth_lead_r20"] = feat["growth_r20"] - feat["main_r20"]
    feat["growth_lead_r60"] = feat["growth_r60"] - feat["main_r60"]
    feat["growth_strength_gap"] = feat["growth_ma20_bias"] - feat["main_ma20_bias"]
    feat["growth_breadth_gap"] = feat["growth_up_frac_20"] - feat["main_up_frac_20"]
    feat["siphon_headwind"] = feat["growth_lead_r20"].clip(lower=0) * (1 - feat["main_up_frac_20"].clip(0, 1))
    feat["rotation_stress"] = feat["growth_lead_r5"].clip(lower=0) * feat["cross_dispersion_r1"]
    feat["growth_vol_headwind"] = feat["growth_lead_r20"].clip(lower=0) * (feat["growth_vol_20"] - feat["main_vol_20"])
    feat["headwind_score_raw"] = feat[["siphon_headwind", "rotation_stress", "growth_vol_headwind"]].sum(axis=1, min_count=1)

    for col in [
        "style_spread_r1", "style_spread_r5", "style_spread_r60", "risk_on_r1", "risk_on_r5",
        "breadth_up_ratio_1", "breadth_up_ratio_5", "cross_dispersion_r1", "main_ma20_bias", "main_ma60_bias",
        "growth_ma20_bias", "growth_ma60_bias", "growth_lead_r5", "growth_lead_r20", "growth_strength_gap",
        "siphon_headwind", "rotation_stress", "headwind_score_raw",
    ]:
        if col in feat.columns:
            feat[f"{col}_chg5"] = feat[col].diff(5)
    return feat.sort_values("trade_date").reset_index(drop=True)


def select_model_features(feat: pd.DataFrame) -> List[str]:
    preferred = [
        "000001__r1", "000001__r5", "000001__r20", "000001__r60", "000001__vol_5", "000001__vol_20", "000001__vol_60",
        "000001__ma20_bias", "000001__ma60_bias", "000001__liq_z_20", "000001__ma20_over_ma60", "000001__up_frac_10",
        "000001__up_frac_20", "000001__dist_from_high_20", "399001__r1", "399001__r5", "399001__r20", "399001__r60",
        "399001__vol_5", "399001__vol_20", "399001__vol_60", "399001__ma20_bias", "399001__ma60_bias", "399001__liq_z_20",
        "000905__r1", "000905__r5", "000905__r20", "000905__r60", "000905__vol_5", "000905__vol_20", "000905__vol_60",
        "000905__ma20_bias", "000905__ma60_bias", "000905__liq_z_20", "000852__r1", "000852__r5", "000852__r20", "000852__r60",
        "000852__vol_5", "000852__vol_20", "000852__vol_60", "000852__ma20_bias", "000852__ma60_bias", "000852__liq_z_20",
        "399006__r1", "399006__r5", "399006__r20", "399006__r60", "399006__vol_5", "399006__vol_20", "399006__vol_60",
        "399006__ma20_bias", "399006__ma60_bias", "399006__liq_z_20", "000688__r1", "000688__r5", "000688__r20", "000688__r60",
        "000688__vol_5", "000688__vol_20", "000688__vol_60", "000688__ma20_bias", "000688__ma60_bias", "000688__liq_z_20",
        "main_r1", "main_r5", "main_r20", "main_r60", "growth_r1", "growth_r5", "growth_r20", "growth_r60",
        "style_spread_r1", "style_spread_r5", "style_spread_r20", "style_spread_r60", "style_spread_ma20_bias", "style_spread_ma60_bias",
        "risk_on_r1", "risk_on_r5", "risk_on_r20", "risk_on_r60", "breadth_up_ratio_1", "breadth_up_ratio_5", "breadth_trend_ratio",
        "cross_dispersion_r1", "cross_dispersion_r5", "main_ma20_bias", "main_ma60_bias", "growth_ma20_bias", "growth_ma60_bias",
        "main_up_frac_10", "main_up_frac_20", "main_vol_shock", "vol_spread_vol_5", "vol_spread_vol_20", "vol_spread_vol_60",
        "399006_minus_000001_r1", "399006_minus_000001_r5", "399006_minus_000001_r20", "000688_minus_000001_r1", "000688_minus_000001_r5",
        "000688_minus_000001_r20", "000905_minus_000001_r1", "000905_minus_000001_r5", "000905_minus_000001_r20",
        "000852_minus_000001_r1", "000852_minus_000001_r5", "000852_minus_000001_r20", "style_spread_r1_chg5", "style_spread_r5_chg5",
        "style_spread_r60_chg5", "risk_on_r1_chg5", "risk_on_r5_chg5", "breadth_up_ratio_1_chg5", "breadth_up_ratio_5_chg5",
        "cross_dispersion_r1_chg5", "main_ma20_bias_chg5", "main_ma60_bias_chg5", "growth_ma20_bias_chg5", "growth_ma60_bias_chg5",
        "growth_lead_r5", "growth_lead_r20", "growth_lead_r60", "growth_strength_gap", "growth_breadth_gap", "siphon_headwind",
        "rotation_stress", "growth_vol_headwind", "headwind_score_raw", "growth_lead_r5_chg5", "growth_lead_r20_chg5",
        "growth_strength_gap_chg5", "siphon_headwind_chg5", "rotation_stress_chg5", "headwind_score_raw_chg5",
    ]
    return [c for c in preferred if c in feat.columns]


In [4]:
def build_return_table(ret_df: pd.DataFrame) -> pd.DataFrame:
    ret_use = ret_df.copy()
    ret_use["trade_date"] = ret_use["date"].astype(int)
    ret_use["ret"] = ret_use["0"].astype(float)
    return ret_use[["trade_date", "ret"]].sort_values("trade_date").reset_index(drop=True)


def build_feature_matrix(df_raw: pd.DataFrame, feature_cols: Optional[List[str]] = None) -> Tuple[pd.DataFrame, List[str]]:
    feat = build_market_feature_table(df_raw).sort_values("trade_date").reset_index(drop=True)
    feature_cols = select_model_features(feat) if feature_cols is None else [c for c in feature_cols if c in feat.columns]
    feat = feat.copy()
    feat.loc[:, feature_cols] = feat.loc[:, feature_cols].shift(1)
    return feat, feature_cols


def build_strategy_return_features(ret_use: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    rs = ret_use[["trade_date", "ret"]].copy().sort_values("trade_date").reset_index(drop=True)
    rs["ret_lag1"] = rs["ret"]
    for w in [3, 5, 10, 20, 40, 60]:
        rs[f"ret_r{w}"] = rs["ret"].rolling(w, min_periods=w).sum()
    for w in [5, 20, 60]:
        rs[f"ret_ewm{w}"] = rs["ret"].ewm(span=w, adjust=False).mean()
        rs[f"ret_vol{w}"] = rs["ret"].rolling(w, min_periods=w).std()
    for w in [20, 60]:
        rs[f"ret_skew{w}"] = rs["ret"].rolling(w, min_periods=w).skew()
    rs["ret_win5"] = rs["ret"].gt(0).rolling(5, min_periods=5).mean()
    rs["ret_win20"] = rs["ret"].gt(0).rolling(20, min_periods=20).mean()
    rs["ret_z20"] = rolling_zscore(rs["ret"], 20, 10)
    rs["ret_z60"] = rolling_zscore(rs["ret"], 60, 20)
    nav = (1 + rs["ret"].fillna(0)).cumprod()
    rs["ret_dd20"] = nav / nav.rolling(20, min_periods=5).max() - 1
    rs["ret_dd60"] = nav / nav.rolling(60, min_periods=20).max() - 1
    rs["ret_dd120"] = nav / nav.rolling(120, min_periods=40).max() - 1
    rs["ret_bad_streak5"] = rs["ret"].lt(0).rolling(5, min_periods=5).sum()
    rs["ret_bad_streak10"] = rs["ret"].lt(0).rolling(10, min_periods=10).sum()
    feature_cols = [c for c in rs.columns if c not in {"trade_date", "ret"}]
    rs.loc[:, feature_cols] = rs.loc[:, feature_cols].shift(1)
    return rs[["trade_date"] + feature_cols], feature_cols


def add_strategy_return_features(feat_lagged: pd.DataFrame, ret_use: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    rs_feat, rs_cols = build_strategy_return_features(ret_use)
    feat = feat_lagged.merge(rs_feat, on="trade_date", how="left")
    interaction_specs = [
        ("risk_on_r5", "ret_r5", "risk_on_x_ret_r5"),
        ("risk_on_r20", "ret_r20", "risk_on_x_ret_r20"),
        ("style_spread_r5", "ret_r5", "style_x_ret_r5"),
        ("breadth_up_ratio_5", "ret_dd20", "breadth_x_dd20"),
        ("cross_dispersion_r1", "ret_vol20", "disp_x_ret_vol20"),
        ("main_vol_shock", "ret_dd20", "shock_x_dd20"),
        ("main_ma60_bias", "ret_dd60", "trend_x_dd60"),
        ("siphon_headwind", "ret_dd20", "headwind_x_dd20"),
        ("growth_lead_r20", "ret_r20", "growthlead_x_ret_r20"),
    ]
    interaction_cols = []
    for left, right, out_col in interaction_specs:
        if left in feat.columns and right in feat.columns:
            feat[out_col] = feat[left] * feat[right]
            interaction_cols.append(out_col)
    return feat, rs_cols + interaction_cols


def summarize_feature_availability(feat_lagged: pd.DataFrame, feature_cols: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame, int]:
    rows = []
    for col in feature_cols:
        first_valid = feat_lagged.loc[feat_lagged[col].notna(), "trade_date"]
        if first_valid.empty:
            continue
        index_code = None
        if "__" in col:
            prefix = col.split("__", 1)[0]
            if prefix.isdigit() and len(prefix) == 6:
                index_code = prefix
        rows.append({"feature": col, "index_code": index_code, "first_valid_date": int(first_valid.iloc[0])})
    feature_start = pd.DataFrame(rows).sort_values(["first_valid_date", "feature"]).reset_index(drop=True)
    index_start = (
        feature_start[feature_start["index_code"].notna()]
        .groupby("index_code", as_index=False)
        .agg(first_usable_date=("first_valid_date", "max"), used_feature_count=("feature", "count"))
        .sort_values(["first_usable_date", "index_code"]) 
        .reset_index(drop=True)
    )
    model_start_date = int(feature_start["first_valid_date"].max())
    return feature_start, index_start, model_start_date


In [5]:
def fit_label_spec(ret_series: pd.Series, method: str, upper_q: float = 0.67, lower_q: float = 0.33, upper_thresh: float = 0.006, lower_thresh: float = -0.003) -> Dict[str, Any]:
    valid = ret_series.dropna()
    spec = {
        "method": method,
        "upper_q": upper_q,
        "lower_q": lower_q,
        "upper_thresh": upper_thresh,
        "lower_thresh": lower_thresh,
    }
    if method == "ternary_quantile":
        spec["q_lo"] = float(valid.quantile(lower_q))
        spec["q_hi"] = float(valid.quantile(upper_q))
    return spec


def apply_label_spec(ret_series: pd.Series, spec: Dict[str, Any]) -> pd.Series:
    s = ret_series.copy()
    labels = pd.Series(np.nan, index=s.index, dtype=float)
    if spec["method"] == "ternary_quantile":
        labels.loc[s.notna()] = 1
        labels.loc[s >= spec["q_hi"]] = 2
        labels.loc[s <= spec["q_lo"]] = 0
    elif spec["method"] == "ternary_threshold":
        labels.loc[s.notna()] = 1
        labels.loc[s > spec["upper_thresh"]] = 2
        labels.loc[s < spec["lower_thresh"]] = 0
    else:
        raise ValueError(f"Unknown method: {spec['method']}")
    return labels.astype("Int64")


def prepare_feature_block(feat: pd.DataFrame, feature_cols: List[str], fill_values: Optional[pd.Series] = None) -> Tuple[pd.DataFrame, pd.Series]:
    x = feat[["trade_date"] + feature_cols].copy().sort_values("trade_date").reset_index(drop=True)
    x.loc[:, feature_cols] = x.loc[:, feature_cols].ffill()
    if fill_values is None:
        fill_values = x.loc[:, feature_cols].median(numeric_only=True)
    x.loc[:, feature_cols] = x.loc[:, feature_cols].fillna(fill_values)
    x = x.dropna(subset=feature_cols).reset_index(drop=True)
    return x, fill_values


def build_classifier(clf_type: str = "lgbm_tuned"):
    if clf_type == "lgbm_tuned" and HAS_LGBM:
        return LGBMClassifier(
            n_estimators=300,
            learning_rate=0.02,
            num_leaves=7,
            max_depth=3,
            min_child_samples=60,
            subsample=0.9,
            colsample_bytree=0.6,
            reg_alpha=0.5,
            reg_lambda=2.0,
            class_weight="balanced",
            random_state=42,
            verbose=-1,
        )
    if clf_type == "lgbm" and HAS_LGBM:
        return LGBMClassifier(
            n_estimators=120,
            learning_rate=0.03,
            num_leaves=15,
            max_depth=3,
            min_child_samples=40,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_alpha=0.2,
            reg_lambda=1.0,
            class_weight="balanced",
            random_state=42,
            verbose=-1,
        )
    if clf_type == "logistic":
        return LogisticRegression(max_iter=1000, C=0.3, random_state=42)
    return GradientBoostingClassifier(n_estimators=120, learning_rate=0.03, max_depth=2, subsample=0.8, random_state=42)


def fit_supervised_model(feat_lagged: pd.DataFrame, ret_use: pd.DataFrame, feature_cols: List[str], start_date: int, train_end: int, label_method: str, clf_type: str, upper_q: float = 0.67, lower_q: float = 0.33, upper_thresh: float = 0.006, lower_thresh: float = -0.003) -> Dict[str, Any]:
    fit_feat = feat_lagged[(feat_lagged["trade_date"] >= start_date) & (feat_lagged["trade_date"] <= train_end)].copy()
    fit_ret = ret_use[(ret_use["trade_date"] >= start_date) & (ret_use["trade_date"] <= train_end)].copy()
    x_fit, fill_values = prepare_feature_block(fit_feat, feature_cols)
    x_fit = x_fit.merge(fit_ret, on="trade_date", how="inner")
    label_spec = fit_label_spec(x_fit["ret"], method=label_method, upper_q=upper_q, lower_q=lower_q, upper_thresh=upper_thresh, lower_thresh=lower_thresh)
    y_fit = apply_label_spec(x_fit["ret"], label_spec)
    valid = y_fit.notna()
    scaler = StandardScaler()
    X_fit = scaler.fit_transform(x_fit.loc[valid, feature_cols].values)
    y = y_fit.loc[valid].astype(int).values
    clf = build_classifier(clf_type)
    clf.fit(X_fit, y)
    return {
        "model": clf,
        "scaler": scaler,
        "fill_values": fill_values,
        "label_spec": label_spec,
        "feature_cols": feature_cols,
        "train_frame": x_fit.loc[valid].copy().reset_index(drop=True),
        "train_end": train_end,
        "last_label_date": int(ret_use["trade_date"].max()),
    }


def predict_full_sample(feat_lagged: pd.DataFrame, ret_use: pd.DataFrame, fit_pack: Dict[str, Any], start_date: int) -> pd.DataFrame:
    feature_cols = fit_pack["feature_cols"]
    extra_cols = [c for c in ["headwind_score_raw", "growth_lead_r20", "growth_strength_gap", "growth_breadth_gap", "siphon_headwind", "rotation_stress"] if c in feat_lagged.columns]
    full_feat = feat_lagged[feat_lagged["trade_date"] >= start_date].copy().reset_index(drop=True)
    x_full, _ = prepare_feature_block(full_feat, feature_cols, fit_pack["fill_values"])
    X_full = fit_pack["scaler"].transform(x_full[feature_cols].values)
    pred = fit_pack["model"].predict(X_full)
    out = x_full[["trade_date"] + extra_cols].copy()
    out["state"] = pred.astype(int)
    out["state_name"] = out["state"].map({0: "bad", 1: "neutral", 2: "good"})
    if hasattr(fit_pack["model"], "predict_proba"):
        prob = fit_pack["model"].predict_proba(X_full)
        for i in range(prob.shape[1]):
            out[f"prob_{i}"] = prob[:, i]
    out = out.merge(ret_use, on="trade_date", how="left")
    out["ret_label"] = apply_label_spec(out["ret"], fit_pack["label_spec"])
    out["has_label"] = out["ret"].notna()
    out["is_after_train_end"] = out["trade_date"] > fit_pack["train_end"]
    out["is_after_last_label_date"] = out["trade_date"] > fit_pack["last_label_date"]
    return out.sort_values("trade_date").reset_index(drop=True)


def feature_importance_table(fit_pack: Dict[str, Any]) -> pd.DataFrame:
    clf = fit_pack["model"]
    feature_cols = fit_pack["feature_cols"]
    if hasattr(clf, "feature_importances_"):
        imp = clf.feature_importances_
    elif hasattr(clf, "coef_"):
        imp = np.abs(np.asarray(clf.coef_)).mean(axis=0)
    else:
        return pd.DataFrame(columns=["feature", "importance"])
    return pd.DataFrame({"feature": feature_cols, "importance": imp}).sort_values("importance", ascending=False).reset_index(drop=True)


def score_states(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, float]]:
    perf = summarize_strategy_by_state(df, state_col="state", ret_col="ret")
    if perf.empty:
        return perf, {"n_states": 0, "min_state_frac": np.nan, "best_mean_ret": np.nan, "worst_mean_ret": np.nan}
    return perf, {
        "n_states": int(len(perf)),
        "min_state_frac": float(perf["count_pct"].min()),
        "best_mean_ret": float(perf["mean_ret"].max()),
        "worst_mean_ret": float(perf["mean_ret"].min()),
    }


def walk_forward_oof(feat_lagged: pd.DataFrame, ret_use: pd.DataFrame, feature_cols: List[str], start_date: int, config: Dict[str, Any], n_folds: int = OOF_FOLDS) -> pd.DataFrame:
    dates = sorted(ret_use.loc[(ret_use["trade_date"] >= start_date) & (ret_use["trade_date"] <= TRAIN_END), "trade_date"].unique())
    folds = np.array_split(dates, n_folds)
    preds = []
    for fold_dates in folds:
        if len(fold_dates) == 0:
            continue
        history = [d for d in dates if d < fold_dates[0]]
        if not history:
            continue
        fit_pack = fit_supervised_model(
            feat_lagged=feat_lagged,
            ret_use=ret_use,
            feature_cols=feature_cols,
            start_date=start_date,
            train_end=max(history),
            **config,
        )
        fold_pred = predict_full_sample(feat_lagged, ret_use, fit_pack, start_date)
        preds.append(fold_pred[fold_pred["trade_date"].isin(fold_dates)].copy())
    return pd.concat(preds, axis=0, ignore_index=True).sort_values("trade_date").reset_index(drop=True)


In [6]:
df_raw = pd.read_csv(INDEX_PATH)
df_raw["trade_date"] = df_raw["trade_date"].astype(int)
df_raw["idx"] = df_raw["idx"].astype(int)

df_ret = pd.read_csv(RETURN_PATH)
df_ret["date"] = df_ret["date"].astype(int)

df_ret_use = build_return_table(df_ret)
feat_base, base_feature_cols = build_feature_matrix(df_raw)
base_fit = fit_supervised_model(
    feat_lagged=feat_base,
    ret_use=df_ret_use,
    feature_cols=base_feature_cols,
    start_date=20200206,
    train_end=TRAIN_END,
    label_method="ternary_quantile",
    upper_q=0.67,
    lower_q=0.33,
    clf_type="lgbm",
)
base_top_features = feature_importance_table(base_fit).head(30)["feature"].tolist()
feat_lagged, strategy_feature_cols = add_strategy_return_features(feat_base, df_ret_use)
headwind_feature_cols = [
    c for c in [
        "growth_lead_r5", "growth_lead_r20", "growth_lead_r60", "growth_strength_gap", "growth_breadth_gap",
        "siphon_headwind", "rotation_stress", "growth_vol_headwind", "headwind_score_raw",
        "growth_lead_r5_chg5", "growth_lead_r20_chg5", "growth_strength_gap_chg5",
        "siphon_headwind_chg5", "rotation_stress_chg5", "headwind_score_raw_chg5",
    ] if c in feat_lagged.columns
]
feature_cols = list(dict.fromkeys([c for c in base_top_features + strategy_feature_cols + headwind_feature_cols if c in feat_lagged.columns]))
_, index_start_info, model_start_date = summarize_feature_availability(feat_lagged, feature_cols)

print(f"index rows = {len(df_raw):,}, date range = {df_raw['trade_date'].min()} ~ {df_raw['trade_date'].max()}")
print(f"return rows = {len(df_ret):,}, date range = {df_ret['date'].min()} ~ {df_ret['date'].max()}")
print(f"model_start_date = {model_start_date}")
print(f"selected feature count = {len(feature_cols)}")
display(index_start_info)


index rows = 27,117, date range = 20140102 ~ 20260528
return rows = 1,702, date range = 20190102 ~ 20260107
model_start_date = 20200206
selected feature count = 76


,index_code,first_usable_date,used_feature_count
0,399006,20140110,2
1,000001,20140116,2
2,000852,20140130,3
3,399001,20140130,2
4,000688,20200206,3


In [7]:
train_oof = walk_forward_oof(
    feat_lagged=feat_lagged,
    ret_use=df_ret_use,
    feature_cols=feature_cols,
    start_date=model_start_date,
    config=FINAL_CONFIG,
)

final_fit = fit_supervised_model(
    feat_lagged=feat_lagged,
    ret_use=df_ret_use,
    feature_cols=feature_cols,
    start_date=model_start_date,
    train_end=TRAIN_END,
    **FINAL_CONFIG,
)
full_pred = predict_full_sample(
    feat_lagged=feat_lagged,
    ret_use=df_ret_use,
    fit_pack=final_fit,
    start_date=model_start_date,
)

validation_oos = full_pred[
    (full_pred["trade_date"] >= HOLDOUT_START)
    & (full_pred["trade_date"] <= HOLDOUT_END)
    & full_pred["has_label"]
].copy().reset_index(drop=True)

train_perf, train_score = score_states(train_oof)
validation_perf, validation_score = score_states(validation_oos)

print("train OOF score:", train_score)
display(train_perf.round(6))
print("validation score:", validation_score)
display(validation_perf.round(6))


train OOF score: {'n_states': 3, 'min_state_frac': 0.19015659955257272, 'best_mean_ret': 0.007000985894275713, 'worst_mean_ret': -0.001331679263723852}


,state,count,count_pct,mean_ret,std_ret,tstat,sharpe_ann,win_rate,max_drawdown
0,0,170,0.190157,-0.001332,0.013291,-1.306342,-1.558619,0.423529,-0.309840
1,1,464,0.519016,0.001522,0.009573,3.425376,2.473756,0.540948,-0.123628
2,2,260,0.290828,0.007001,0.017961,6.285207,6.063740,0.703846,-0.168005


validation score: {'n_states': 3, 'min_state_frac': 0.2839506172839506, 'best_mean_ret': 0.006898443570432763, 'worst_mean_ret': -0.0017496252362010592}


,state,count,count_pct,mean_ret,std_ret,tstat,sharpe_ann,win_rate,max_drawdown
0,0,69,0.283951,-0.001750,0.011633,-1.249291,-2.339629,0.420290,-0.163284
1,1,104,0.427984,0.002853,0.009787,2.972871,4.534893,0.596154,-0.035673
2,2,70,0.288066,0.006898,0.012254,4.710125,8.757722,0.771429,-0.018806


In [8]:
print("train OOF tests:")
train_tests = state_return_hypothesis_tests(train_oof)
print({k: v for k, v in train_tests.items() if k != "pairwise"})
display(train_tests["pairwise"].round(6))

print("2025 holdout tests:")
validation_tests = state_return_hypothesis_tests(validation_oos)
print({k: v for k, v in validation_tests.items() if k != "pairwise"})
display(validation_tests["pairwise"].round(6))

cm_df = validation_oos[["state", "ret_label"]].dropna().copy()
if len(cm_df):
    y_true = cm_df["ret_label"].astype(int)
    y_pred = cm_df["state"].astype(int)
    used_labels = sorted(set(y_true) | set(y_pred))
    display(pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=used_labels),
        index=[f"true_{x}" for x in used_labels],
        columns=[f"pred_{x}" for x in used_labels],
    ))
    print(classification_report(y_true, y_pred, zero_division=0))


train OOF tests:
{'anova_p': 1.1905396416774582e-10, 'kruskal_p': 3.837539248068272e-12}


,state_i,state_j,mean_i,mean_j,mean_diff,t_stat,p_value
0,0,1,-0.001332,0.001522,-0.002854,-2.566378,0.010894
1,0,2,-0.001332,0.007001,-0.008333,-5.518559,0.000000
2,1,2,0.001522,0.007001,-0.005479,-4.568386,0.000007


2025 holdout tests:
{'anova_p': 3.837484713182169e-05, 'kruskal_p': 0.00011903440166125966}


,state_i,state_j,mean_i,mean_j,mean_diff,t_stat,p_value
0,0,1,-0.001750,0.002853,-0.004603,-2.711055,0.007626
1,0,2,-0.001750,0.006898,-0.008648,-4.267630,0.000037
2,1,2,0.002853,0.006898,-0.004045,-2.310221,0.022509


,pred_0,pred_1,pred_2
true_0,26,21,9
true_1,34,59,37
true_2,9,24,24


              precision    recall  f1-score   support

           0       0.38      0.46      0.42        56
           1       0.57      0.45      0.50       130
           2       0.34      0.42      0.38        57

    accuracy                           0.45       243
   macro avg       0.43      0.45      0.43       243
weighted avg       0.47      0.45      0.45       243



In [9]:
export_pred = full_pred.copy()
oof_cols = ["trade_date", "state", "state_name"] + [c for c in ["prob_0", "prob_1", "prob_2"] if c in train_oof.columns]
export_pred = export_pred.merge(train_oof[oof_cols].rename(columns={c: f"oof_{c}" for c in oof_cols if c != "trade_date"}), on="trade_date", how="left")
for col in ["state", "state_name", "prob_0", "prob_1", "prob_2"]:
    oof_col = f"oof_{col}"
    if oof_col in export_pred.columns:
        base = export_pred[col] if col in export_pred.columns else pd.Series(np.nan, index=export_pred.index)
        export_pred[col] = export_pred[oof_col].combine_first(base)
export_pred = export_pred.drop(columns=[c for c in export_pred.columns if c.startswith("oof_")])

train_oof_dates = set(train_oof["trade_date"].tolist())
export_pred["sample_split"] = np.select(
    [
        export_pred["trade_date"].isin(train_oof_dates),
        (export_pred["trade_date"] <= TRAIN_END) & (~export_pred["trade_date"].isin(train_oof_dates)),
        (export_pred["trade_date"] >= HOLDOUT_START) & (export_pred["trade_date"] <= HOLDOUT_END) & export_pred["has_label"],
        (export_pred["trade_date"] > HOLDOUT_END) & export_pred["has_label"],
        ~export_pred["has_label"],
    ],
    ["train_oof", "train_warmup_fit", "holdout_2025", "future_labeled", "future_unlabeled"],
    default="other",
)

export_pred.to_csv(OUTPUT_PATH, index=False)
print(f"saved to: {OUTPUT_PATH}")
print(export_pred[["sample_split", "state"]].value_counts().sort_index())

display(feature_importance_table(final_fit).head(20))

diagnostic = export_pred[
    ((export_pred["trade_date"] >= 20260302) & (export_pred["trade_date"] <= 20260313))
    | ((export_pred["trade_date"] >= 20260520) & (export_pred["trade_date"] <= 20260528))
].copy()
display(diagnostic[[c for c in ["trade_date", "sample_split", "state", "state_name", "prob_0", "prob_1", "prob_2", "headwind_score_raw", "growth_lead_r20", "growth_strength_gap", "growth_breadth_gap"] if c in diagnostic.columns]])


saved to: classifier1_full_sample_predictions.csv
sample_split      state
future_labeled    1.0        2
                  2.0        1
future_unlabeled  1.0       65
                  2.0       26
holdout_2025      0.0       69
                  1.0      104
                  2.0       70
train_oof         0.0      170
                  1.0      464
                  2.0      260
train_warmup_fit  0.0       78
                  1.0       95
                  2.0      126
Name: count, dtype: int64


,feature,importance
0,ret_lag1,88
1,000905_minus_000001_r1,82
2,000001__dist_from_high_20,76
3,disp_x_ret_vol20,76
4,ret_z20,73
5,cross_dispersion_r1,60
6,risk_on_r60,59
7,399006__ma20_bias,58
8,style_spread_ma60_bias,56
9,000852_minus_000001_r20,55


,trade_date,sample_split,state,state_name,prob_0,prob_1,prob_2,headwind_score_raw,growth_lead_r20,growth_strength_gap,growth_breadth_gap
1470,20260302,future_unlabeled,1.0,neutral,0.237094,0.526083,0.236822,0.000000,-0.030421,-0.024542,-0.150000
1471,20260303,future_unlabeled,1.0,neutral,0.217915,0.492113,0.289972,0.000000,-0.027778,-0.027793,-0.166667
1472,20260304,future_unlabeled,1.0,neutral,0.202214,0.431639,0.366146,0.000000,-0.027606,-0.025149,-0.166667
1473,20260305,future_unlabeled,1.0,neutral,0.274083,0.432656,0.293261,0.000000,-0.038320,-0.028089,-0.175000
1474,20260306,future_unlabeled,1.0,neutral,0.214378,0.406280,0.379342,0.000000,-0.026555,-0.019475,-0.133333
1475,20260309,future_unlabeled,1.0,neutral,0.257147,0.439600,0.303253,0.000000,-0.024746,-0.020945,-0.125000
1476,20260310,future_unlabeled,1.0,neutral,0.234441,0.477148,0.288411,0.000000,-0.045743,-0.020624,-0.166667
1477,20260311,future_unlabeled,1.0,neutral,0.255485,0.485476,0.259039,0.000067,-0.039234,-0.008484,-0.166667
1478,20260312,future_unlabeled,1.0,neutral,0.215844,0.507830,0.276326,0.000157,-0.028905,-0.004070,-0.166667
1479,20260313,future_unlabeled,1.0,neutral,0.196373,0.586136,0.217492,0.000009,-0.027521,-0.009162,-0.133333
